### classifier, torch

In [1]:
import torch

In [3]:
from datasets import load_dataset

ds = load_dataset('imdb', split=['train[:10%]', 'test[:10%]'])

In [76]:
import nltk

tokenizer = nltk.tokenize.WordPunctTokenizer()

train = [tokenizer.tokenize(text.lower()) for text in ds[0]['text']]

In [94]:
train_labels = [label for label in ds[0]['label']]

In [77]:
PAD_TOKEN = '<pad>'

train_seq_len = 0
for t in train:
    if len(t) > train_seq_len:
        train_seq_len = len(t)

train = [text + [PAD_TOKEN] * (train_seq_len - len(text)) for text in train]

In [78]:
vocab_set = set()
for t in train:
    for w in t:
        vocab_set.add(w)

In [79]:
vocab_i_map = {}
i_vocab_map = {}
for i, token in enumerate(vocab_set):
    vocab_i_map[token] = i
    i_vocab_map[i] = token

In [80]:
for i, text in enumerate(train):
    for j, token in enumerate(text):
        train[i][j] = vocab_i_map[token]

In [82]:
vocab_size = len(vocab_set)
emb_dim = 64
inp_size = 512
hid_size = 1024

In [119]:
class Cls(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
        self.embs = torch.nn.Embedding(vocab_size, emb_dim)
        self.fc1 = torch.nn.Linear(emb_dim, hid_size)
        self.activation1 = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hid_size, 1)
        self.activation2 = torch.nn.Sigmoid()
        
        self.norm = torch.nn.LayerNorm(hid_size)
        self.dropout = torch.nn.Dropout(p=0.2)
        
    # X: [batch_size, seq_len]
    def forward(self, X):
        X = self.embs(X) # [batch_size, seq_len, emb_dim]
        
        X = self.fc1(X) # [batch_size, seq_len, hid_size]
        X = self.activation1(X)
        
        X = self.norm(X)
        
        X = self.dropout(X)
        
        X = self.fc2(X) # [batch_size, seq_len, 2]
        X = self.activation2(X)
    
        return X


In [120]:
dummy_batch = torch.tensor(train[:2])

In [123]:
model = Cls()

In [125]:
opt = torch.optim.Adam(model.parameters())

In [126]:
batch_size = 16

train_loss = []

for i in range(0, len(train), batch_size):
    if i + batch_size > len(train):
        break
    
    batch = train[i : i + batch_size]
    batch = torch.tensor(batch)
    
    labels = train_labels[i : i + batch_size]
    labels = torch.tensor(labels)
    
    preds = model.forward(batch)
    preds = preds.squeeze(dim=2)
    
    batch_train_loss = torch.nn.functional.cross_entropy(preds, labels)
    train_loss.append(batch_train_loss)
    
    print(batch_train_loss)
    
    batch_train_loss.backward()
    opt.step()
    opt.zero_grad()
    
    

tensor(7.6603, grad_fn=<NllLossBackward0>)
tensor(7.3346, grad_fn=<NllLossBackward0>)
tensor(7.1303, grad_fn=<NllLossBackward0>)
tensor(6.9766, grad_fn=<NllLossBackward0>)
tensor(7.0463, grad_fn=<NllLossBackward0>)
tensor(7.0250, grad_fn=<NllLossBackward0>)
tensor(6.8389, grad_fn=<NllLossBackward0>)
tensor(6.8672, grad_fn=<NllLossBackward0>)
tensor(6.9800, grad_fn=<NllLossBackward0>)
tensor(6.8640, grad_fn=<NllLossBackward0>)
tensor(6.8523, grad_fn=<NllLossBackward0>)
tensor(6.8359, grad_fn=<NllLossBackward0>)
tensor(6.8705, grad_fn=<NllLossBackward0>)
tensor(6.8178, grad_fn=<NllLossBackward0>)
tensor(6.8563, grad_fn=<NllLossBackward0>)
tensor(6.9019, grad_fn=<NllLossBackward0>)
tensor(6.8066, grad_fn=<NllLossBackward0>)
tensor(6.8444, grad_fn=<NllLossBackward0>)
tensor(6.8687, grad_fn=<NllLossBackward0>)
tensor(6.8545, grad_fn=<NllLossBackward0>)
tensor(6.7243, grad_fn=<NllLossBackward0>)
tensor(6.7883, grad_fn=<NllLossBackward0>)
tensor(6.7945, grad_fn=<NllLossBackward0>)
tensor(6.84